# Solutions · Chapter 00-04 · Prediction, explanation, and cause

Worked answers with reasoning. E7 and E14 both produce results that contradict the instinct
"control for everything you have", and they contradict it in *opposite directions* - those two
are the ones to read slowly.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# The chapter's data, rebuilt exactly.
rng = np.random.default_rng(3)
n_customers = 2000
loyalty = rng.normal(0, 1, n_customers)
p_email = 1 / (1 + np.exp(-1.5 * loyalty))
emailed = rng.random(n_customers) < p_email
noise = rng.normal(0, 8, n_customers)
TRUE_EFFECT = 5.0
spend = 40 + 15 * loyalty + TRUE_EFFECT * emailed + noise
customers = pd.DataFrame({"loyalty": loyalty, "emailed": emailed.astype(int), "spend": spend})
print("naive coefficient:", round(LinearRegression().fit(customers[["emailed"]], customers["spend"]).coef_[0], 2))

## E1 · What is a confounder?

> A confounder is a variable that influences **both** who receives the treatment and the
> outcome, so that a plain comparison of treated with untreated mixes the effect of the treatment
> with the effect of the confounder.

Loyalty qualifies on both counts by construction: `p_email` is a function of loyalty, so it
drives who is emailed; and `spend` has a `15 * loyalty` term, so it drives the outcome. Remove
either arrow and the naive comparison would be fine - which is exactly what E6 demonstrates.

**The trap:** calling any correlated variable a confounder. A variable correlated with the
outcome alone is just a useful feature. Both arrows are required.

## E2 · The three questions

1. **Prediction** - what will Y be for this row? Answered by a fitted model, natively.
2. **Description / explanation** - what did the model use, and how? Answered by a fitted model,
   with care about wording; the honest form is "the model relies on X", not "X drives Y".
3. **Causation** - what happens to Y if I set X? **Not answered by a fitted model.** It requires
   an intervention, or assumptions about how the world works that you state explicitly and that
   the data cannot verify.

**The instinct:** the third question does not announce itself. It arrives as "so we should do
more X", and the model's output looks the same either way.

## E3 · Great predictor, useless lever

`emailed` predicts spend well because the marketing system only sent the email to loyal
customers - so knowing that someone was emailed tells you they are probably loyal, and loyal
people spend more. The variable is carrying *information about loyalty*.

Now pull the lever. Email everybody, and `emailed` is 1 for everyone: it no longer distinguishes
anybody, and the information it was carrying is gone. What remains is the email's own 5 EUR.

Stated generally: **a predictive feature earns its value from how it varies in the data you
observed. An intervention changes that variation, and can destroy exactly the thing that made
the feature useful.** The prediction was never wrong; it was answering a question about the world
as it was, not about a world you have changed.

## E4 · The coupon, by hand

**Naive:**
- Coupon (P, Q, S): `(60 + 70 + 30) / 3 = 160 / 3 = 53.33`
- No coupon (R, T, U): `(55 + 20 + 25) / 3 = 100 / 3 = 33.33`
- Difference: **20.00 EUR**

**Within region:**
- City with coupon (P, Q): `(60 + 70) / 2 = 65.00`; city without (R): `55.00` -> **10.00**
- Rural with coupon (S): `30.00`; rural without (T, U): `(20 + 25) / 2 = 22.50` -> **7.50**
- Average of the two: `(10.00 + 7.50) / 2 = **8.75 EUR**`

**Which number to report: 8.75**, and the sentence to attach is the important half:

> "Comparing like with like by region, the coupon is associated with about 8.75 EUR more spend.
> This is not proof that the coupon *caused* it - region may not be the only thing that differs
> between who got a coupon and who did not."

**Why the naive number is inflated:** two of the three coupons went to city customers, who spend
more regardless. The coupon variable is partly acting as a label for "city".

**The trap this catches:** reporting 8.75 as though adjusting for one variable settled the
matter. Adjusting for region makes the estimate *better*, not *correct*. Only the caveat makes
the report honest.

## E5 · The business case, both versions

In [ ]:
naive_coef = LinearRegression().fit(customers[["emailed"]], customers["spend"]).coef_[0]
n_new, cost_each = 3500, 0.40

for label, effect in [("promised (naive 20.42)", naive_coef), ("real (true effect 5.00)", TRUE_EFFECT)]:
    revenue = n_new * effect
    print(f"{label:<26} revenue {revenue:9,.0f}  cost {n_new * cost_each:7,.0f}  profit {revenue - n_new * cost_each:9,.0f} EUR")

print(f"\nbreak-even true effect per customer: {cost_each:.2f} EUR")

- Promised: `3,500 x 20.42 = 71,465` revenue, minus `1,400` cost -> **70,065 EUR profit**
- Real: `3,500 x 5.00 = 17,500` revenue, minus `1,400` cost -> **16,100 EUR profit**

**Break-even is a true effect of 0.40 EUR per customer** - the cost of sending one email.

Now notice the awkward thing, because it is the real-world lesson: **the campaign is still very
profitable.** 16,100 EUR is a good outcome, and the true effect of 5.00 clears the 0.40 hurdle
sixteen times over. Nobody will call this a failure.

That is precisely why the error survives. The decision happened to be right; the *number* was
wrong by 4x, and it will be used again to size the next campaign, to compare against a different
channel, and to justify a budget. **A causal error that does not reverse the decision is the kind
that never gets caught** - and the next decision it informs may be much closer to the line.

## E6 · Turning selection off

In [ ]:
def simulate(selection_strength, seed=3, n=2000):
    r = np.random.default_rng(seed)
    L = r.normal(0, 1, n)
    p = 1 / (1 + np.exp(-selection_strength * L))
    E = r.random(n) < p
    S = 40 + 15 * L + TRUE_EFFECT * E + r.normal(0, 8, n)
    return pd.DataFrame({"loyalty": L, "emailed": E.astype(int), "spend": S})

print(f"true effect is {TRUE_EFFECT:.2f} EUR throughout\n")
for strength in (0.0, 0.5, 1.5, 3.0):
    d = simulate(strength)
    naive = LinearRegression().fit(d[["emailed"]], d["spend"]).coef_[0]
    adj = LinearRegression().fit(d[["emailed", "loyalty"]], d["spend"]).coef_[0]
    print(f"selection strength {strength:<4} naive {naive:6.2f}   adjusted {adj:5.2f}")

| selection strength | naive | adjusted |
|---|---|---|
| 0.0 | **4.36** | 5.11 |
| 0.5 | 12.14 | 5.01 |
| 1.5 | 20.42 | 5.39 |
| 3.0 | 24.97 | 4.91 |

**At strength 0.0 the naive estimate is already right** (4.36 against 5.00, the gap being
ordinary noise). Nothing was adjusted, no confounder was handled, and the simple comparison
works - because when the email is assigned independently of loyalty, the emailed and
not-emailed groups have the same average loyalty, and the thing that was contaminating the
comparison is gone.

**That is exactly what randomisation does, and this table is the argument for it.** A coin flip
sets the selection strength to zero not just for loyalty but for *every* variable at once,
including the ones you never measured, never recorded, and never thought of. No modelling
technique can do that, because modelling can only handle variables that exist in your table.

The adjusted column stays near 5 throughout, which is the good news - and E7 is the bad news
about how fragile that column is.

## E7 · "We controlled for it"

In [ ]:
d = simulate(1.5)
rng_proxy = np.random.default_rng(11)

print(f"true effect {TRUE_EFFECT:.2f}; unadjusted estimate "
      f"{LinearRegression().fit(d[['emailed']], d['spend']).coef_[0]:.2f}\n")
for sd in (0.0, 0.5, 1.0, 2.0):
    noisy = d.assign(recorded_loyalty=d["loyalty"] + rng_proxy.normal(0, sd, len(d)))
    coef = LinearRegression().fit(noisy[["emailed", "recorded_loyalty"]], noisy["spend"]).coef_[0]
    print(f"loyalty measured with noise sd {sd:<4} -> estimate {coef:5.2f}")

| noise in the recorded loyalty | estimate |
|---|---|
| 0.0 (perfect measurement) | 5.39 |
| 0.5 | 9.32 |
| 1.0 | 14.50 |
| 2.0 | 18.06 |

The true effect is 5.00 and the unadjusted estimate is 20.42. **A loyalty score measured with as
much noise as signal (sd 1.0, when loyalty itself has sd 1.0) recovers barely a third of the
correction.** At sd 2.0 you are at 18.06 - almost as wrong as not adjusting at all.

This is called **residual confounding**, and it is the reason the phrase *"we controlled for
it"* deserves a follow-up question rather than a nod. Controlling for a variable removes the bias
that variable causes **only to the extent that you measured it accurately**. In real data,
constructs like loyalty, engagement, severity, socioeconomic status and intent are proxied by
crude measurements, and the correction is correspondingly partial.

**The part that makes it dangerous:** the output looks identical in all four rows. There is no
standard error, no warning, no diagnostic in the model that says "your confounder is noisy".
The only defence is knowing how the variable was measured - which is a data-provenance question,
and is why module 02 spends a whole chapter on it (02-02).

**What to say instead of "we controlled for it":** *"we adjusted for a recorded loyalty score;
if that score is a noisy measure of actual loyalty, the estimate is biased upward, and we do not
know by how much."*

## E8 · "App users churn 60% less"

**The confounder to look for first: engagement, or intent to stay.** People who were already
committed to the product are the people who bother to install the app. The app did not make them
loyal; their loyalty made them install the app. It is the chapter's situation with the labels
changed.

**The comparison you would want instead:** among customers who look *equally likely to install*
- same tenure, same usage, same plan, same support history - compare those who did install with
those who did not. Better still, compare people who *were offered* the app prompt with people who
were not, whatever they subsequently did.

**What would actually settle it:** randomise the prompt. Show the app invitation to a random half
of eligible customers and compare 30-day churn between the two *assigned* groups - including the
people who were invited and ignored it. Comparing installers with non-installers inside the
invited group would reintroduce exactly the bias you set out to remove, because who chooses to
install is not random.

**The number to be suspicious of:** "60% less" is very large. Effects that big from a low-cost
intervention are usually selection. A useful reflex: *the more dramatic the observational
effect, the more likely it is measuring who chose the treatment.*

## E9 · The painkiller that predicts death

Ordered from most to least likely:

1. **Confounding by indication - overwhelmingly the most likely.** The drug is given to patients
   who are in more pain, more advanced disease, or post-surgery. Sicker people receive it, and
   sicker people die more. The drug is a marker of severity, not a cause of death. This explains
   the vast majority of such findings.
2. **Reverse timing in the data.** The prescription may be recorded near the end of life -
   palliative care, comfort measures - so the drug appears in the record *because* the patient was
   dying. The model has learned a consequence of the outcome and is using it as a predictor. This
   is target leakage wearing a clinical coat, and 04-05 covers it.
3. **A genuine adverse effect.** Possible, and it does happen - but it is the third hypothesis,
   not the first, and establishing it needs a trial or a carefully designed observational study,
   not a coefficient.

**What I would ask the clinicians first:** *"who gets prescribed this, and when in the course of
care?"* One sentence from a doctor about the prescribing protocol usually distinguishes
hypotheses 1 and 2 immediately, and it is available in an afternoon. The instinct being trained
is that **the fastest route to a causal explanation is asking how the treatment gets assigned**,
not fitting another model.

## E10 · "R-squared 0.91, so we understand what drives sales"

> R-squared measures how well the model reproduces variation in sales; it says nothing about
> what would happen if we changed any of the inputs. A model can predict beautifully using
> variables that are consequences of sales rather than causes - number of delivery vans dispatched
> predicts sales superbly, and hiring more vans sells nothing. So the statement can be entirely
> true and entirely useless at the same time: excellent forecasts, and no valid guidance about
> which lever to pull. If the question is what *drives* sales, we need either an experiment or an
> explicit causal argument about which variables could have been set differently.

**What the interviewer is listening for:** whether you can hold "the model is good" and "the
conclusion is invalid" in your head simultaneously. Candidates who attack the R-squared have
missed the point; the model is fine.

## E11 · The tutorial that already shipped to everyone

**Approach 1: compare before and after, in time.** Take retention for users who joined shortly
before the tutorial launched and shortly after.
*Assumption:* nothing else about those two cohorts or about the product changed at the same
moment. That is a strong assumption and usually false - launches come with marketing pushes,
seasonality, and other releases. It gets more credible if you can find a **comparison group** that
did not receive the tutorial across the same dates (another region, another platform) and check
that the two groups moved in parallel *before* the launch.

**Approach 2: randomise going forward.** The tutorial has shipped, but you can withhold or vary
it for a random subset of *new* users from today.
*Assumption:* new users respond like the users you care about, and that withholding is acceptable
to the business and to the users. This is the stronger evidence, and the cost is time plus
someone agreeing to a holdout.

**A third worth naming:** if the tutorial rolled out by a threshold - to accounts created after
a specific timestamp, or above a usage cut-off - then users just either side of that line are
nearly comparable, and comparing them is a much stronger design than a general before/after.
That family of methods is mapped in 12-07.

**And the answer that is always available:** report the association with the assumption stated
and the uncertainty attached. "Retention is 4 points higher after launch; we cannot separate the
tutorial from the March pricing change" is a useful, honest sentence.

## E12 · Prediction, description, or causation?

| Request | Which | What it needs |
|---|---|---|
| (a) Which customers should we call this week? | **Prediction** (then a decision rule) | Rank by predicted risk or value; no causal claim needed to *prioritise* - though see the caveat below |
| (b) Does calling customers increase renewals? | **Causation** | An experiment: randomise who gets called. Comparing called with uncalled will be badly confounded, because reps call the accounts they think are worth calling |
| (c) Which factors best predict renewal? | **Description** | A model plus honest wording. Legitimate as stated; becomes causal the moment someone says "so let's change one" |
| (d) If we raise the price by 5%, how many customers do we lose? | **Causation**, and the hardest kind | Past price changes are confounded with everything happening around them. Needs an experiment, a natural experiment, or an explicitly stated demand model |
| (e) Which customers are most similar to the ones who left? | **Description**, unsupervised | A similarity measure. Says nothing about why they left or what would retain them |

**The caveat on (a):** it looks purely predictive, but if you call the high-risk customers and
they renew, next year's data shows "customers we called renewed more" - and you have manufactured
exactly the confounding of (b). Acting on predictions changes the data you collect. Watch for
this whenever a model's output influences the world it will next be trained on; 11-07 covers
feedback loops.

## E13 · Explaining it to Tomás

> The 20 EUR is real - the people who got the email really did spend more. But the machine only
> sent it to your best customers, so we compared your best customers against everyone else and
> called the difference "the email". It is like noticing that people carrying umbrellas get wet
> less and concluding umbrellas stop rain. Sending it to everyone gains about 5 EUR each, not 20.

(72 words.)

**Where the umbrella comparison stops being accurate:** with umbrellas the causal story is
obvious to everyone, so nobody is fooled. Here the email plausibly *could* cause more spending -
it is a sensible mechanism - which is why the wrong conclusion is believable and gets acted on.
Adding that sentence is what makes the explanation honest rather than merely clever.

## E14 · When adjusting makes it worse

The chapter's confounder sat *before* the treatment: loyalty caused both the email and the
spending. Now build the opposite shape - a variable *between* the treatment and the outcome.

The email makes people open the app more, and opening the app more makes them spend more. The
email's true **total** effect is therefore its direct effect plus everything it does through the
app.

In [ ]:
r = np.random.default_rng(1)
n = 6000

email = r.random(n) < 0.5                                  # randomised, so no confounding at all
app_opens = 2 + 1.5 * email + r.normal(0, 1, n)            # the email causes app opens
spend2 = 30 + 4 * app_opens + 1.0 * email + r.normal(0, 5, n)   # both routes lead to spend

mediation = pd.DataFrame({"emailed": email.astype(int), "app_opens": app_opens, "spend": spend2})

print("true direct effect      : 1.00 EUR")
print("true effect via the app : 4.00 x 1.50 = 6.00 EUR")
print("true TOTAL effect       : 7.00 EUR\n")
print(f"not adjusting for app_opens: {LinearRegression().fit(mediation[['emailed']], mediation['spend']).coef_[0]:.2f} EUR")
print(f"adjusting for app_opens    : {LinearRegression().fit(mediation[['emailed', 'app_opens']], mediation['spend']).coef_[0]:.2f} EUR")

**Not adjusting: 6.97. Adjusting: 0.97.** The true total effect is 7.00.

Here the *simpler* model is the correct one, and adding a variable destroyed the answer.

**Why.** `app_opens` is not a confounder, it is a **mediator** - it lies on the causal path from
the email to the spending. Controlling for it means asking "what does the email do *among people
with the same number of app opens*", which deliberately switches off the main route by which the
email works. The 0.97 is not wrong as a quantity; it is the correct **direct** effect. It is
simply not the number that answers "should we send the email", because sending the email is
exactly what increases app opens.

**The lesson, and it is why causality cannot be automated:**

> Whether to adjust for a variable depends on **where that variable sits in the causal story**,
> and no amount of looking at the data can tell you that.

Both `loyalty` and `app_opens` are correlated with the treatment and with the outcome. They look
identical in the table. Adjusting for one is essential and adjusting for the other is a mistake,
and the only way to know which is which is to reason about what causes what - from domain
knowledge, not from the numbers.

So "control for everything you have" is not a safe default. It is a coin flip between fixing the
estimate and destroying it. Chapter 12-07 gives you the vocabulary (colliders belong on this list
too, and they are worse than both) and 07-06 revisits it when interpreting feature importances,
where the same confusion shows up wearing different clothes.

---

## Where to go next

Back to the chapter for the mastery check and flashcards. That finishes **module 00**.

Next is **02-01 · What is a row?**, opening module 02 on data literacy - or the optional Python
bridge first, whose diagnostic is **01-01**.